# GRPO-Zero on Colab — train Qwen2.5 on GSM8K with RLVR

From-scratch **GRPO** (Group Relative Policy Optimization) + a verifiable reward (RLVR) on GSM8K.
Repo: https://github.com/Vivek-Chaudhari30/GRPO-Zero

**Before running:** `Runtime > Change runtime type > GPU` (T4 is enough for 0.5B; L4/A100 for 1.5B).

Pipeline: verify env → unit tests → overfit sanity (M4) → baseline eval → full GRPO train (M5) → eval trained policy + curves (M6).

In [ ]:
!nvidia-smi

In [ ]:
# Clone the repo and install deps (Colab already ships a CUDA torch — don't reinstall it).
!git clone https://github.com/Vivek-Chaudhari30/GRPO-Zero.git
%cd GRPO-Zero
!pip install -q transformers datasets peft accelerate trl pyyaml matplotlib

In [ ]:
# Sanity: the verifier + GRPO-math unit tests must pass on the box too.
!python -m pytest -q tests/

## M4 — Overfit sanity check
Fix 16 prompts and train only on them. The loop is working if **mean reward climbs** and **KL grows from ~0** (it starts at 0 because the LoRA adapter initializes to zero, so the policy begins identical to the reference).

In [ ]:
!python -m src.train --overfit 16 --steps 30 --group-size 8 --prompts-per-step 8 \
    --max-new-tokens 512 --lr 5e-5 --save-dir checkpoints/overfit \
    --log results/overfit_log.json
!python scripts/plot_curves.py results/overfit_log.json results/overfit_curve.png

## Baseline — base model pass@1 on the full GSM8K test set
This is the number the trained policy must beat. (~1319 problems; tens of minutes on T4.)

In [ ]:
!python -m src.eval --config configs/grpo_gsm8k.yaml --n 1319 --tag baseline_full \
    --save-completions results/baseline_completions_full.json

## M5 — Full GRPO training run
Trains the LoRA policy on the GSM8K train split. Tune `--steps` / `--group-size` to your GPU and time budget (T4 is slower than L4/A100). Checkpoints land in `checkpoints/grpo_lora`.

In [ ]:
!python -m src.train --config configs/grpo_gsm8k.yaml \
    --save-dir checkpoints/grpo_lora --log results/train_log.json
!python scripts/plot_curves.py results/train_log.json results/reward_curve.png

## M6 — Eval the trained policy + before/after
Same eval, now with the trained LoRA adapter loaded.

In [ ]:
!python -m src.eval --config configs/grpo_gsm8k.yaml --n 1319 --tag trained \
    --adapter checkpoints/grpo_lora --save-completions results/trained_completions.json

In [ ]:
import json
from IPython.display import Image, display

m = json.load(open('results/metrics.json'))
b = m.get('baseline_full', m.get('baseline'))
t = m.get('trained')
print(f"baseline pass@1 = {b['accuracy']}  (n={b['n']})")
if t:
    print(f"trained  pass@1 = {t['accuracy']}  (n={t['n']})")
    print(f"delta           = {t['accuracy'] - b['accuracy']:+.4f}")
display(Image('results/reward_curve.png'))

## Save results back (optional)
Commit the real numbers + curves to the repo so the README can cite them.
```python
# from google.colab import files; files.download('results/reward_curve.png')
# or: configure git creds and push results/ + checkpoints metadata
```